# Netflix Content Strategy Analysis (2016-2021)

## Thesis
How did Netflix's content strategy evolve from 2016 to 2021,
and did COVID-19 change what they produced?

## Dataset
- Source: Kaggle Netflix Movies and TV Shows dataset
- 8,807 titles spanning 2008-2021
- Note: 9% of titles are missing country data and were excluded from country analysis

In [64]:
import pandas as pd 
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

netflix_data = pd.read_csv('netflix_titles.csv')

## Section 1 - Setting the scene
A first look at the data: total titles, content type, and data quality notes.

In [65]:
netflix_data.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [66]:
netflix_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [67]:
netflix_data.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [68]:
# dropping missing values for descriptive variables: director, cast, and country
# dropping missing values since we will explore this in this analysis
netflix_data['director'] = netflix_data['director'].fillna('Unknown')
netflix_data['cast'] = netflix_data['cast'].fillna('Unknown')
netflix_data['country'] = netflix_data['country'].fillna('Unknown')
netflix_data = netflix_data.dropna(subset=['date_added'])

In [69]:
netflix_data['genre'] = netflix_data['listed_in'].str.split(', ')

In [70]:
netflix_data['date_added'] = pd.to_datetime(netflix_data['date_added'].str.strip())

In [71]:
netflix_data['year_added'] = netflix_data['date_added'].dt.year.astype(int)

In [72]:
netflix_data.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,genre,year_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",[Documentaries],2021
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...","[International TV Shows, TV Dramas, TV Mysteries]",2021
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,"[Crime TV Shows, International TV Shows, TV Ac...",2021
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...","[Docuseries, Reality TV]",2021
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,"[International TV Shows, Romantic TV Shows, TV...",2021


## Section 2 - Growth over time
How many titles did Netflix add each year between 2016 to 2021? Did movies or TV shows grow faster?

In [73]:
netflix_filtered = netflix_data[netflix_data['year_added'].between(2016,2021)].copy()
netflix_filtered['era'] = netflix_filtered['year_added'].apply(
    lambda x: 'pre-covid' if x <= 2019 else 'post-covid'
)
netflix_filtered.groupby(['year_added','type'])['title'].count().unstack().plot()

<Axes: xlabel='year_added'>

### Key insight
Netflix's content library grew significantly year over year, with movies consistently outnumbering TV shows. Growth peaked in 2019 before slowing slightly in 2020, likely due to COVID-19 production shutdown

## Section 3 - Genre shift
Which genre dominated before covid (2016 - 2019) vs after (2020-2021)

In [74]:
netflix_exploded = netflix_filtered.explode('genre')
netflix_exploded.groupby(['era','genre'])['title'].count()

era         genre                   
post-covid  Action & Adventure          366
            Anime Features               37
            Anime Series                 86
            British TV Shows             74
            Children & Family Movies    292
                                       ... 
pre-covid   TV Sci-Fi & Fantasy          37
            TV Shows                      9
            TV Thrillers                 28
            Teen TV Shows                31
            Thrillers                   328
Name: title, Length: 84, dtype: int64

In [75]:
top_genres = (netflix_exploded
              .groupby(['era','genre'])['title']
              .count()
              .reset_index()
              .rename(columns = {'title':'count'})
              .sort_values(['era','count'], ascending=[True,False])
              .groupby('era')
              .head(3)
              )
top_genres

,era,genre,count
16,post-covid,International Movies,983
12,post-covid,Dramas,947
7,post-covid,Comedies,715
58,pre-covid,International Movies,1755
54,pre-covid,Dramas,1453
49,pre-covid,Comedies,942


In [76]:
# count titles per era per genre
genre_count = (netflix_exploded
               .groupby(['era','genre'])['title']
               .count()
               .reset_index()
               .rename(columns ={'title':'count'})
               )

# divide by number of years in each era to get average per year 
genre_count['years'] = genre_count['era'].map({
    'pre-covid': 4, 
    'post-covid': 2
})

genre_count['avg_per_year'] = genre_count['count'] / genre_count['years']

top_genres = (genre_count 
              .sort_values(['era','avg_per_year'], ascending=[True,False])
              .groupby('era')
              .head(3)
              )
print(top_genres)

           era                 genre  count  years  avg_per_year
16  post-covid  International Movies    983      2        491.50
12  post-covid                Dramas    947      2        473.50
7   post-covid              Comedies    715      2        357.50
58   pre-covid  International Movies   1755      4        438.75
54   pre-covid                Dramas   1453      4        363.25
49   pre-covid              Comedies    942      4        235.50


In [77]:
pivot_df = top_genres.pivot(index = 'genre', columns = 'era', values ='avg_per_year')
pivot_df['pct_change'] = ((pivot_df['post-covid'] - pivot_df['pre-covid']) / pivot_df['pre-covid'] *100).round(1)
pivot_df

era,post-covid,pre-covid,pct_change
genre,,,
Comedies,357.5,235.50,51.8
Dramas,473.5,363.25,30.4
International Movies,491.5,438.75,12.0


### Key insight
After normalizing for time, all three top genre domination both post- and pre- covid actually **increased** post-covid.
Comedies saw the sharpest growth at ~49% more titles per year, suggesting Netflix leaned into lighter content during the pandemic. Internation movies grew ~11%, supporting the thesis that Netflix accelerated its global content push after 2020.

## Section 4 - International push
Did Netflix invest more in non-US content after the pandemic began?

**Note on methodology:** Analysis is filtered to 2016 onwards. Years prior to 2016 were excluded due to insufficient data. For example, 2008-2014 had fewer than 25 titles per year, making percentage comparisions unreliable and potentially misleading.

In [78]:
netflix_filtered['is_us'] = netflix_filtered['country'].str.contains('United States', na = False)
netflix_filtered.groupby(['year_added', 'is_us'])['title'].count().unstack()

is_us,False,True
year_added,,
2016,226,203
2017,726,462
2018,1049,600
2019,1160,856
2020,1051,828
2021,871,627


In [79]:
internation_growth = (netflix_filtered.groupby(['year_added','is_us'])['title']
                      .count()
                      .unstack()
                      .fillna(0)
                      .astype(int)
                      )
internation_growth.columns = ['International','US']
internation_growth

,International,US
year_added,,
2016,226,203
2017,726,462
2018,1049,600
2019,1160,856
2020,1051,828
2021,871,627


### Initial approach - era averages
Looking at averages per era gives a high level overview, however averaging across multiple years can mask important year-by-year tends.

In [80]:
internation_growth['total'] = internation_growth['International'] + internation_growth['US']
internation_growth['International_pct'] = (
    (internation_growth['International'] / internation_growth['total'] *100)
).round(1)
print(internation_growth)

            International   US  total  International_pct
year_added                                              
2016                  226  203    429               52.7
2017                  726  462   1188               61.1
2018                 1049  600   1649               63.6
2019                 1160  856   2016               57.5
2020                 1051  828   1879               55.9
2021                  871  627   1498               58.1


In [81]:
pre = internation_growth.loc[2016:2019, 'International_pct'].mean().round(1)
post = internation_growth.loc[2020:2021, 'International_pct'].mean().round(1)
change = (post - pre).round(1)

print(f"Pre-COVID avg internal %: {pre}%")
print (f"Post-COVID avg international %: {post}%")
print(f"Change: {change}%")

Pre-COVID avg internal %: 58.7%
Post-COVID avg international %: 57.0%
Change: -1.7%


### Refined approach - year by year breakdown
Breaking this down annually reveals a more nuanced three-act story that averages alone would have hidden.

In [82]:
print(internation_growth.loc[2016:2021, 'International_pct'])

year_added
2016    52.7
2017    61.1
2018    63.6
2019    57.5
2020    55.9
2021    58.1
Name: International_pct, dtype: float64


### Key insight
International content grew steadily from 52.7% in 2016 to a peak of 63.6% in 2018, suggesting Netflix was aggressively expanding globally during this period. However, this trend revered in 2019 and dipped further in 2020 to 55.9%, likely refelcting worldwide production shutdowns during the pandemic. The slight recovery to 58.1% in 2021 suggest Netflix's international strategy was beginning to bounce back as production resumed. 

## Section 5 - Content maturity 
DId Netflix shift towards more mature content rating post-COVID?

*Note on methodology*: Analysis is filtered to 2016-2021 using `netflix_filtered` to ensure consistency with previous section. Counts are normalized by years (pre-covid = 4 years, post-covid = 2 years) to allow fair comparison.

In [83]:
rating_counts = (netflix_filtered.groupby(['era','rating'])['title']
                 .count()
                 .unstack(level = 0)
                 .fillna(0)
                 .astype(int)
                 )

# normalize by years per era
rating_counts['pre_covid_avg'] = (rating_counts['pre-covid']/4).round(1)
rating_counts['post_covid_avg'] = (rating_counts['post-covid']/2).round(1)

# calculate percent change

rating_counts['pct_change'] = (
    (rating_counts['post_covid_avg'] - rating_counts['pre_covid_avg']) / rating_counts['pre_covid_avg'] *100
).round(1)

rating_counts

# top 3 per era
rating_summary = (rating_counts[['pre_covid_avg','post_covid_avg','pct_change']]
                  .sort_values('post_covid_avg',ascending=False)
                  ).head(5)
print(rating_summary)

era     pre_covid_avg  post_covid_avg  pct_change
rating                                           
TV-MA           498.5           580.0        16.3
TV-14           342.2           382.5        11.8
R               104.2           189.0        81.4
PG-13            55.0           134.0       143.6
TV-PG           150.0           121.5       -19.0


### Key insight
Netflix's content shifted toward more mature ratings post-COVID. TV-PG content 
declined by 19%, dropping out of the top ratings entirely, suggesting a move away 
from family-friendly content. Meanwhile mature ratings surged; TV-MA grew 16.3%, 
TV-14 grew 11.8%, and most notably R-rated content nearly doubled (+81.4%). However,
the sharpest growth was PG-13, which increased by 143.6%, suggesting 
Netflix aggressively expanded its mid-maturity movie catalog post-COVID.

## Section 6 - Seasonal patterns
Does Netflix follow a content release calender? Did COVID disrupt it?

In [84]:
netflix_filtered['month_added'] = netflix_filtered['date_added'].dt.month

seasonal = (netflix_filtered.groupby(['era','month_added'])['title']
            .count()
            .unstack(level = 0)
            .fillna(0)
)

seasonal['pre_covid_avg'] = (seasonal['pre-covid'] /4).round(1)
seasonal['post_covid_avg'] = (seasonal['post-covid']/2).round(1)

seasonal['pct_change'] = ((seasonal['post_covid_avg'] - seasonal['pre_covid_avg']) / seasonal['pre_covid_avg'] *100).round(1)

seasonal


era,post-covid,pre-covid,pre_covid_avg,post_covid_avg,pct_change
month_added,,,,,
1,337,397,99.2,168.5,69.9
2,223,332,83.0,111.5,34.3
3,249,487,121.8,124.5,2.2
4,365,392,98.0,182.5,86.2
5,289,335,83.8,144.5,72.4
6,363,358,89.5,181.5,102.8
7,403,416,104.0,201.5,93.8
8,307,444,111.0,153.5,38.3
9,351,408,102.0,175.5,72.1


In [85]:
seasonal_melted = seasonal[['pre_covid_avg','post_covid_avg']].reset_index()

fig = px.line(seasonal_melted, 
              x ='month_added', 
              y =['pre_covid_avg','post_covid_avg'],
              title = 'Monthly content additions: pre vs post COVID',
              labels = {'month_added': 'Month',
                        'value': 'Avg title added',
                        'variable': 'Era'
                        }, 
            color_discrete_map= {
                'pre_covid_avg':'#221F1F',
                'post_covid_avg': '#E50914'
            }
        )

newnames = {'pre_covid_avg':'Pre-COVID', 'post_covid_avg': 'Post-COVID'}
fig.for_each_trace(lambda t: t.update(name =newnames[t.name]))
fig.show()

In [86]:
(seasonal.get('post_covid_avg').loc[3:9].sum() - seasonal.get('pre_covid_avg').loc[3:9].sum()).round()

np.float64(453.0)

In [87]:
abs(seasonal.get('post_covid_avg').loc[10:12].sum() - seasonal.get('pre_covid_avg').loc[10:12].sum()).round()

np.float64(183.0)

## Key insight
Between March and September, Netflix released 453 more content post-COVID than content rleased during pre-COVID, with greatest amount released in July. Interestingly, the narative switches towards the end of the year with pre-Covid content releasing 183 more content than content released post_COVID, suggesting Netflix's release calender drastically fliped trends due to COVID. 

## Section 7 - Summary & conclusions
Key findings from the analysis.